In [0]:
df = spark.table("default.refined_mtc_dataset").toPandas()

print("Shape:", df.shape)
print("\nTarget distribution:")
print(df["target"].value_counts())

Shape: (1107, 197)

Target distribution:
target
0    814
1    293
Name: count, dtype: int64


In [0]:
import numpy as np
import pandas as pd

identifier_columns = [
    "dataset_folder",
    "mtc_file",
    "segment_id",
    "start",
    "end",
    "raw_label"
]

target_column = "target"
group_column = "dataset_folder"

X = df.drop(columns=identifier_columns + [target_column])
y = df[target_column]
groups = df[group_column]

X = X.select_dtypes(include=np.number)

print("Feature matrix:", X.shape)
print("Target:", y.shape)
print("Number of experiments:", groups.nunique())

Feature matrix: (1107, 190)
Target: (1107,)
Number of experiments: 24


In [0]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain experiments:", groups_train.nunique())
print("Test experiments:", groups_test.nunique())

print("\nExperiment overlap:")
print(set(groups_train).intersection(set(groups_test)))

print("\nTrain target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

Train shape: (942, 190)
Test shape: (165, 190)

Train experiments: 19
Test experiments: 5

Experiment overlap:
set()

Train target distribution:
target
0    745
1    197
Name: count, dtype: int64

Test target distribution:
target
1    96
0    69
Name: count, dtype: int64


In [0]:
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [0]:
EXPERIMENT_NAME = "/Shared/cnc_milling_anomaly_detection"

mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow experiment:", EXPERIMENT_NAME)

2026/07/25 18:33:00 INFO mlflow.tracking.fluent: Experiment with name '/Shared/cnc_milling_anomaly_detection' does not exist. Creating a new experiment.


MLflow experiment: /Shared/cnc_milling_anomaly_detection


In [0]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "f1_score": f1_score(
            y_test,
            predictions,
            zero_division=0
        )
    }

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X_test)[:, 1]
        metrics["roc_auc"] = roc_auc_score(
            y_test,
            probabilities
        )

    return predictions, metrics

In [0]:
models = {
    "decision_tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

In [0]:
results = []

for model_name, classifier in models.items():

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", classifier)
    ])

    with mlflow.start_run(run_name=model_name) as run:

        pipeline.fit(X_train, y_train)

        predictions, metrics = evaluate_model(
            pipeline,
            X_test,
            y_test
        )

        mlflow.log_params(classifier.get_params())
        mlflow.log_metrics(metrics)

        mlflow.log_param(
            "number_of_features",
            X_train.shape[1]
        )

        mlflow.log_param(
            "train_rows",
            X_train.shape[0]
        )

        mlflow.log_param(
            "test_rows",
            X_test.shape[0]
        )

        mlflow.log_param(
            "split_type",
            "group_shuffle_split"
        )

        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="model",
            input_example=X_train.head(3)
        )

        results.append({
            "model": model_name,
            "run_id": run.info.run_id,
            **metrics
        })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        by=["f1_score", "recall"],
        ascending=False
    )
)

2026/07/25 18:33:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-700d842f-f782.cloud.databricks.com/ml/experiments/2857151153212128/models/m-48cae9966c5044a3b4cee451dde45b37?o=7474646333398352
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for 

model,run_id,accuracy,precision,recall,f1_score,roc_auc
decision_tree,b5ee6033feee49b6a3355903b5256cc9,0.8606060606060606,0.8411214953271028,0.9375,0.8866995073891626,0.8463164251207729
gradient_boosting,901f807b1d2d4b768dbf989123d70a60,0.806060606060606,0.7711864406779662,0.9479166666666666,0.8504672897196262,0.9390096618357487
random_forest,5714859f560b4d3eadd63763cc0e5e00,0.7454545454545455,0.717741935483871,0.9270833333333334,0.8090909090909091,0.8965881642512077


In [0]:
import mlflow
import pandas as pd

experiment = mlflow.get_experiment_by_name(
    "/Shared/cnc_milling_anomaly_detection"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

metric_columns = [
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc"
]

baseline_comparison = runs[
    metric_columns
].copy()

baseline_comparison.columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC"
]

baseline_comparison = (
    baseline_comparison
    .dropna(subset=["F1 Score"])
    .sort_values("F1 Score", ascending=False)
)

display(baseline_comparison)

Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
decision_tree,0.8606060606060606,0.8411214953271028,0.9375,0.8866995073891626,0.8463164251207729
gradient_boosting,0.806060606060606,0.7711864406779662,0.9479166666666666,0.8504672897196262,0.9390096618357487
improved_random_forest_v2,0.7454545454545455,0.717741935483871,0.9270833333333334,0.8090909090909091,0.9002113526570048
random_forest,0.7454545454545455,0.717741935483871,0.9270833333333334,0.8090909090909091,0.8965881642512077


In [0]:
baseline_only = baseline_comparison[
    baseline_comparison["Model"].isin([
        "decision_tree",
        "random_forest",
        "gradient_boosting"
    ])
].copy()

baseline_only = baseline_only[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ]
].round(4)

print(
    baseline_only
    .sort_values("F1 Score", ascending=False)
    .to_string(index=False)
)

            Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
    decision_tree    0.8606     0.8411  0.9375    0.8867   0.8463
gradient_boosting    0.8061     0.7712  0.9479    0.8505   0.9390
    random_forest    0.7455     0.7177  0.9271    0.8091   0.8966


In [0]:
for model_name, classifier in models.items():

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    print("\n" + "=" * 60)
    print(model_name.upper())
    print("=" * 60)

    print(classification_report(
        y_test,
        predictions,
        digits=4
    ))

    print("Confusion matrix:")
    print(confusion_matrix(
        y_test,
        predictions
    ))


DECISION_TREE
              precision    recall  f1-score   support

           0     0.8966    0.7536    0.8189        69
           1     0.8411    0.9375    0.8867        96

    accuracy                         0.8606       165
   macro avg     0.8688    0.8456    0.8528       165
weighted avg     0.8643    0.8606    0.8583       165

Confusion matrix:
[[52 17]
 [ 6 90]]

RANDOM_FOREST
              precision    recall  f1-score   support

           0     0.8293    0.4928    0.6182        69
           1     0.7177    0.9271    0.8091        96

    accuracy                         0.7455       165
   macro avg     0.7735    0.7099    0.7136       165
weighted avg     0.7644    0.7455    0.7293       165

Confusion matrix:
[[34 35]
 [ 7 89]]

GRADIENT_BOOSTING
              precision    recall  f1-score   support

           0     0.8936    0.6087    0.7241        69
           1     0.7712    0.9479    0.8505        96

    accuracy                         0.8061       165
   ma

In [0]:
best_baseline = (
    results_df
    .sort_values(
        by=["f1_score", "recall", "roc_auc"],
        ascending=False
    )
    .iloc[0]
)

print("Best baseline model:")
print(best_baseline)

Best baseline model:
model                           decision_tree
run_id       b5ee6033feee49b6a3355903b5256cc9
accuracy                             0.860606
precision                            0.841121
recall                                 0.9375
f1_score                               0.8867
roc_auc                              0.846316
Name: 0, dtype: object


In [0]:
best_run_id = best_baseline["run_id"]
best_model_name = best_baseline["model"]

print("Best model:", best_model_name)
print("Best run ID:", best_run_id)

Best model: decision_tree
Best run ID: b5ee6033feee49b6a3355903b5256cc9


In [0]:
REGISTERED_MODEL_NAME = "cnc_milling_anomaly_model"

model_uri = f"runs:/{best_run_id}/model"

model_version = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

print("Registered model:", REGISTERED_MODEL_NAME)
print("Version:", model_version.version)

Successfully registered model 'workspace.default.cnc_milling_anomaly_model'.
2026/07/25 18:35:50 WARNING mlflow.tracking._model_registry.fluent: Run with id b5ee6033feee49b6a3355903b5256cc9 has no artifacts at artifact path 'model', registering model based on models:/m-48cae9966c5044a3b4cee451dde45b37 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Registered model: cnc_milling_anomaly_model
Version: 1


🔗 Created version '1' of model 'workspace.default.cnc_milling_anomaly_model': https://dbc-700d842f-f782.cloud.databricks.com/explore/data/models/workspace/default/cnc_milling_anomaly_model/version/1?o=7474646333398352


In [0]:
from mlflow import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="Champion",
    version=model_version.version
)

print(
    f"Champion alias assigned to version {model_version.version}"
)

Champion alias assigned to version 1


In [0]:
registered = client.get_model_version_by_alias(
    name=REGISTERED_MODEL_NAME,
    alias="Champion"
)

print("Model name:", registered.name)
print("Champion version:", registered.version)
print("Run ID:", registered.run_id)

Model name: workspace.default.cnc_milling_anomaly_model
Champion version: 1
Run ID: b5ee6033feee49b6a3355903b5256cc9


In [0]:
champion_uri = (
    f"models:/{REGISTERED_MODEL_NAME}@Champion"
)

champion_model = mlflow.sklearn.load_model(
    champion_uri
)

sample_predictions = champion_model.predict(
    X_test.head(10)
)

print("Sample predictions:")
print(sample_predictions)

Sample predictions:
[1 1 1 1 1 1 1 1 1 1]


In [0]:
sample_input = X_test.head(1)

prediction = champion_model.predict(sample_input)[0]

if hasattr(champion_model, "predict_proba"):
    probability = champion_model.predict_proba(
        sample_input
    )[0, 1]
else:
    probability = None

print("Prediction:", int(prediction))
print("Meaning:", "Anomaly" if prediction == 1 else "Normal")
print("Anomaly probability:", probability)

Prediction: 1
Meaning: Anomaly
Anomaly probability: 1.0


In [0]:
baseline_metrics = {
    "accuracy": float(best_baseline["accuracy"]),
    "precision": float(best_baseline["precision"]),
    "recall": float(best_baseline["recall"]),
    "f1_score": float(best_baseline["f1_score"]),
    "roc_auc": float(best_baseline["roc_auc"])
}

baseline_metrics

{'accuracy': 0.8606060606060606,
 'precision': 0.8411214953271028,
 'recall': 0.9375,
 'f1_score': 0.8866995073891626,
 'roc_auc': 0.8463164251207729}

In [0]:
baseline_metrics_df = spark.createDataFrame(
    [baseline_metrics]
)

baseline_metrics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.cnc_baseline_metrics")

print("Baseline metrics saved permanently.")

Baseline metrics saved permanently.
